In [11]:
from langgraph.graph import StateGraph , START , END 
from typing import TypedDict 
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [14]:
load_dotenv()

True

In [15]:
llm = HuggingFaceEndpoint(repo_id ="Qwen/Qwen2.5-7B-Instruct")
model = ChatHuggingFace(llm=llm)

In [33]:
class Jokestate(TypedDict):
    
    topic : str
    joke : str
    explanation : str
    

In [ ]:
def generate_joke(state:Jokestate):
    
    prompt = f"generate a joke on the topic {state['topic']}"
    
    response = model.invoke(prompt).content
    
    return {'joke': response}



In [35]:
def genrate_exlanation(state:Jokestate):
    
    prompt = f'Generate an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content
    
    return {'explanation':response}

In [36]:
graph = StateGraph(Jokestate)

graph.add_node('generate_joke',generate_joke)
graph.add_node('genrate_exlanation',genrate_exlanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','genrate_exlanation')
graph.add_edge('genrate_exlanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [38]:
config1 = {"configurable" : {"thread_id" : "1"}}
workflow.invoke({'topic':'pizza'},config = config1)

{'topic': 'pizza',
 'joke': "Sure! Here's a pizza joke for you:\n\nWhy did the pizza go to the doctor?\n\nBecause it was feeling a little crust-aced!",
 'explanation': 'Sure, let\'s break down this pizza joke for you:\n\nThe setup of the joke is a simple scenario: "Why did the pizza go to the doctor?" This immediately sets up an expectation for a reason that might relate to the health or state of the pizza.\n\nThe punchline "Because it was feeling a little crust-aced!" plays on a few wordplay elements:\n1. "Crust-aced" is a play on "stressed." The pizza is experiencing some form of stress or unease, but instead of using the word "stressed," it uses "crust-aced," cleverly combining "crust" (a part of the pizza) with "stressed."\n2. The joke relies on the listener’s ability to connect the usual pizza to an unexpected situation (seeking medical help).\n\nThis type of joke relies heavily on wordplay and the listener\'s ability to make the connection between the pun and the object of the jo

In [39]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Sure! Here's a pizza joke for you:\n\nWhy did the pizza go to the doctor?\n\nBecause it was feeling a little crust-aced!", 'explanation': 'Sure, let\'s break down this pizza joke for you:\n\nThe setup of the joke is a simple scenario: "Why did the pizza go to the doctor?" This immediately sets up an expectation for a reason that might relate to the health or state of the pizza.\n\nThe punchline "Because it was feeling a little crust-aced!" plays on a few wordplay elements:\n1. "Crust-aced" is a play on "stressed." The pizza is experiencing some form of stress or unease, but instead of using the word "stressed," it uses "crust-aced," cleverly combining "crust" (a part of the pizza) with "stressed."\n2. The joke relies on the listener’s ability to connect the usual pizza to an unexpected situation (seeking medical help).\n\nThis type of joke relies heavily on wordplay and the listener\'s ability to make the connection between the pun and t

In [40]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Sure! Here's a pizza joke for you:\n\nWhy did the pizza go to the doctor?\n\nBecause it was feeling a little crust-aced!", 'explanation': 'Sure, let\'s break down this pizza joke for you:\n\nThe setup of the joke is a simple scenario: "Why did the pizza go to the doctor?" This immediately sets up an expectation for a reason that might relate to the health or state of the pizza.\n\nThe punchline "Because it was feeling a little crust-aced!" plays on a few wordplay elements:\n1. "Crust-aced" is a play on "stressed." The pizza is experiencing some form of stress or unease, but instead of using the word "stressed," it uses "crust-aced," cleverly combining "crust" (a part of the pizza) with "stressed."\n2. The joke relies on the listener’s ability to connect the usual pizza to an unexpected situation (seeking medical help).\n\nThis type of joke relies heavily on wordplay and the listener\'s ability to make the connection between the pun and 

In [42]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the pasta break up with the bread?\n\nBecause it couldn't commit to just beingcarbs!",
 'explanation': 'This joke plays on the idea of "carbs," which is a colloquial term for carbohydrates, commonly found in pasta and bread. The humor stems from the double meaning in the punchline: "carbs" can refer both to the type of food and to the romantic relationship aspect of "commitment." The pasta, a carb-heavy food, couldn\'t decide on just being carbs in a relationship with bread, another carb-based food. The play on words suggests that the pasta was too indecisive to commit to a relationship with bread, thus breaking up.'}

In [43]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta break up with the bread?\n\nBecause it couldn't commit to just beingcarbs!", 'explanation': 'This joke plays on the idea of "carbs," which is a colloquial term for carbohydrates, commonly found in pasta and bread. The humor stems from the double meaning in the punchline: "carbs" can refer both to the type of food and to the romantic relationship aspect of "commitment." The pasta, a carb-heavy food, couldn\'t decide on just being carbs in a relationship with bread, another carb-based food. The play on words suggests that the pasta was too indecisive to commit to a relationship with bread, thus breaking up.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17a976-9e5d-6ab0-8002-48f7c4cb4d5c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-08T06:36:56.051780+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint

Fault Tolerence



In [ ]:
import time



# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3 : str
    
    
    # 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"step3": "done"}

# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")
    
    
    
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)



Time travel

In [44]:
workflow.invoke(None,{"configurable" : {'thread_id' : "2" , "checkpoint_id":"1f17a976-869a-6a62-8000-c5b211382536"}})

{'topic': 'pasta',
 'joke': "Why did the pasta go to the doctor?\n\nBecause it wasn't feeling quite al dente!",
 'explanation': 'This joke plays on the double meaning of the term "al dente," which is a culinary term describing pasta that is cooked until it is firm to the bite. In the context of the joke, the pasta is personified and visits a doctor, suggesting that it is not feeling well. The punchline cleverly twists the expectation by using "al dente" both literally and figuratively, as if the pasta is seeking medical attention for a condition related to its texture.'}

Updating State

In [48]:
workflow.update_state({"configurable":{"thread_id":"2","checkpoint_id":"1f17a976-869a-6a62-8000-c5b211382536","checkpoint_ns":""}},{"topic":"samosa"})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f17a9e9-a62d-6cae-8001-1698c7edd3c8'}}

In [49]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17a9e9-a62d-6cae-8001-1698c7edd3c8'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-08T07:28:23.878775+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17a976-869a-6a62-8000-c5b211382536'}}, tasks=(PregelTask(id='2ba97089-19a3-4231-9301-12deb3eadf2b', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the doctor?\n\nBecause it wasn't feeling quite al dente!", 'explanation': 'This joke plays on the double meaning of the term "al dente," which is a culinary term describing pasta that is cooked until it is firm to the bite. In the context of the joke, the pasta is personified and visits a doctor, suggesti

In [50]:
workflow.invoke(None,{"configurable":{"thread_id":"2","checkpoint_id":"1f17a9e9-a62d-6cae-8001-1698c7edd3c8"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa cross the road?\n\nTo get to the curry on the other side!',
 'explanation': 'This joke is a play on the classic "Why did the... cross the road?" setup, which usually involves a character like a chicken, a bear, or more recently, a samosa. The humor comes from the unexpected and delicious twist at the end: instead of a typical animal or object, the subject of the joke (the samosa) is crossing the road to get to some curry. It\'s a clever and somewhat silly way to combine cultural references (samosas being a popular snack, often associated with Indian cuisine) with a familiar joke structure to create a funny and somewhat unexpected punchline!'}